# Songo Stockfish - Multi Colab Minimal

Flux minimal:
1. Monter Drive
2. Préparer / mettre à jour le projet (script centralisé)
3. Générer configs actives par identité Colab (script centralisé)
4. Lancer dataset (génération + build)
5. Entraîner, évaluer, benchmarker, promotion globale

## 1) Monter Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Setup projet + workspace par identité Google Drive

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

GIT_REPO_URL = 'https://github.com/GlennEriss/songo-model-stockfish-for-google-collab.git'
GIT_BRANCH = 'main'
REPO_NAME = 'songo-model-stockfish-for-google-collab'
DRIVE_PROJECT_NAME = 'songo-stockfish'
# Change cette valeur sur chaque Colab: colab_1, colab_2, colab_3, ...
# Si vide, le script tentera une detection auto Drive.
COLAB_IDENTITY = 'colab_1'

WORKTREE = Path(f'/content/{REPO_NAME}')
PYTHON_BIN = sys.executable or 'python3'

# On clone juste assez pour avoir accès aux scripts centralisés.
if not (WORKTREE / '.git').exists():
    if WORKTREE.exists():
        shutil.rmtree(WORKTREE)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, str(WORKTREE)], check=True)

step_script = WORKTREE / 'scripts' / 'colab' / 'notebook_step.py'
summary_path = Path('/tmp/songo_bootstrap_summary.json')
subprocess.run(
    [
        PYTHON_BIN,
        str(step_script),
        'bootstrap',
        '--git-repo-url',
        GIT_REPO_URL,
        '--git-branch',
        GIT_BRANCH,
        '--worktree',
        str(WORKTREE),
        '--drive-project-name',
        DRIVE_PROJECT_NAME,
        '--colab-identity',
        COLAB_IDENTITY,
        '--python-bin',
        PYTHON_BIN,
        '--summary-path',
        str(summary_path),
    ],
    check=True,
)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
DRIVE_ROOT = Path(summary['drive_root'])
DRIVE_IDENTITY_KEY = str(summary.get('drive_identity_key', '')).strip() or 'unknown_drive_identity'
DRIVE_WORKSPACE_ROOT = Path(summary['drive_workspace_root'])

# Propagation kernel notebook pour les cellules suivantes.
env_updates = {
    'SONGO_DRIVE_ROOT': str(DRIVE_ROOT),
    'SONGO_ENFORCE_DRIVE_ROOT_WRITES': '1',
    'SONGO_DRIVE_IDENTITY_KEY': DRIVE_IDENTITY_KEY,
    'SONGO_DRIVE_WORKSPACE_ROOT': str(DRIVE_WORKSPACE_ROOT),
    'SONGO_WORKTREE': str(WORKTREE),
    'SONGO_PYTHON_BIN': PYTHON_BIN,
}
for _k, _v in env_updates.items():
    os.environ[_k] = _v

print('DRIVE_ROOT           =', DRIVE_ROOT)
print('COLAB_IDENTITY       =', COLAB_IDENTITY or '<auto>')
print('DRIVE_IDENTITY_KEY   =', DRIVE_IDENTITY_KEY)
print('DRIVE_WORKSPACE_ROOT =', DRIVE_WORKSPACE_ROOT)
print('WORKTREE             =', WORKTREE)

## 3) Générer configs actives minimalistes

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'generate-configs',
        '--worktree',
        WORKTREE,
    ],
    check=True,
)

## 4) Audit stockage (aucune purge)

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'audit-storage',
    ],
    check=True,
)

## 5) Lancer la génération de dataset (long run, cumulatif)

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'run-job',
        'dataset-generate',
        '--worktree',
        WORKTREE,
        '--heartbeat-seconds',
        '30',
    ],
    check=True,
)

## 6) Lancer le build dataset (long run, cumulatif)

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'run-job',
        'dataset-build',
        '--worktree',
        WORKTREE,
        '--heartbeat-seconds',
        '30',
    ],
    check=True,
)

## 7) Train -> Eval -> Benchmark (promotion globale incluse)

In [ ]:
import os
import subprocess
import sys

WORKTREE = os.environ.get('SONGO_WORKTREE', '/content/songo-model-stockfish-for-google-collab')
PYTHON_BIN = os.environ.get('SONGO_PYTHON_BIN', (sys.executable or 'python3'))
subprocess.run(
    [
        PYTHON_BIN,
        f'{WORKTREE}/scripts/colab/notebook_step.py',
        'run-job',
        'train-eval-benchmark',
        '--worktree',
        WORKTREE,
        '--heartbeat-seconds',
        '30',
    ],
    check=True,
)